---
title: "Tugas 1 Machine Learning"
format:
  html:
    code-fold: true
  ipynb: 
    output-file: tugas1-052747132.ipynb
  pdf:
    code-fold: false
jupyter: python3
number-sections: true
execute: 
  enabled: true
appendix-style: plain
---

# Pendahuluan {#sec-intro}

```yaml
Nama: Muhammad Hidayat
NIM: 052747132
```

Tujuan dari tugas ini adalah untuk mempersiapkan dataset yang akan digunakan dalam proses pemodelan machine learning. Dataset yang digunakan memiliki beberapa permasalahan umum, seperti data yang hilang dan inkonsistensi format data, sehingga memerlukan langkah-langkah preprocessing untuk memastikan kualitas data sebelum digunakan dalam model.

# Persiapan Lingkungan

## Library yang digunakan {#sec-library}

In [ ]:
import numpy as np
import pandas as pd

## Dataset

Dataset yang digunakan, [dataset_tugas1_preprocessing.csv](./dataset_tugas1_preprocessing.csv), berisi informasi mahasiswa dari berbagai program studi. Dataset ini sengaja dirancang dengan beberapa permasalahan umum dalam dunia nyata, seperti data yang hilang (missing values), inkonsistensi format data, serta tipe data kategorikal yang belum siap diproses oleh model machine learning.

Langkah pertama dalam eksplorasi data adalah memuat dataset ke dalam DataFrame menggunakan pandas. 

In [ ]:
#| label: tbl-data-head
#| fig-cap: "Beberapa baris pertama dari dataset"

# Memuat dataset
data = pd.read_csv('dataset_tugas1_preprocessing.csv')

# Menampilkan beberapa baris pertama dari dataset
display(data.head())

# Eksplorasi Data {#sec-eda}

Sebelum melakukan pemodelan, penting untuk memahami struktur dan kualitas data yang kita miliki. Salah satu langkah awal adalah mengetahui struktur dataset, termasuk jumlah baris dan kolom, serta tipe data dari setiap kolom.

Dari @tbl-data-head, terdapat 99 baris data, tiapa baris mewakili satu mahasiswa, dan terdapat 8 kolom yang memberikan informasi tentang mahasiswa tersebut. Berikut adalah penjelasan singkat mengenai setiap kolom:

- **ID**: Merupakan identifier unik untuk setiap mahasiswa. Ini biasanya digunakan oleh Nomor Induk Mahasiswa (NIM). Dalam hal ini, ID berupa format `MHXXX` dimana `XXX` adalah angka yang menunjukkan urutan mahasiswa.
- **Nama**: Merupakan nama lengkap mahasiswa. Ini digunakan sebagai pelengkap dalam identifikasi mahasiswa.
- **Jenis_Kelamin**: Merupakan jenis kelamin mahasiswa, yang dapat berupa 'Laki-laki' atau 'Perempuan'.
- **Prodi**: Merupakan program studi yang diikuti oleh mahasiswa, seperti 'Informatika', 'Teknik Komputer', 'Data Science', dll.
- **Status**: Merupakan status mahasiswa, yang dapat berupa 'Aktif', 'Cuti', 'Lulus', atau 'DO' atau "_dropout_".
- **Nilai_Akhir**: Merupakan nilai akhir mahasiswa dalam bentuk mutu huruf, seperti 'A', 'B', 'C', 'D', atau 'E'. Nilai ini menunjukkan performa akademik mahasiswa, dengan 'A' sebagai performa terbaik dan 'E' sebagai performa terburuk.
- **Tanggal_Ujian**: Merupakan tanggal ujian yang diikuti oleh mahasiswa. Terdapat dua format, yaittu 'YYYY/MM/DD' dan 'DD-MM-YYYY'. Dalam pemodelan, kita perlu memastikan bahwa format tanggal konsisten agar dapat diproses dengan benar.
- **Umur**: Merupakan umur mahasiswa dalam tahun.

In [ ]:
# Menampilkan informasi tentang dataset
display(data.info())

Menggunakan `data.info()`, kita dapat melihat bahwa sebagian besar kolom masih berupa tipe `str` atau teks, termasuk 'Tanggal_Ujian'. Data ini akan diolah menjadi tipe data yang sesuai untuk pemodelan.

In [ ]:
# Analisis statistik deskriptif untuk kolom numerik
numeric_summary = data.describe()
display(numeric_summary)

# Hitung jumlah entri string unik per kolom
unique_entries = data.apply(lambda col: col.nunique())

# Cetak hasilnya
print("Jumlah entri string unik per kolom:")
display(unique_entries)

In [ ]:
# Tampilkan entri string unik untuk setiap kolom
for column in data.columns:
    unique_entries = data[column].unique()
    if len(unique_entries) <= 10:  # Hanya tampilkan jika jumlah entri unik tidak terlalu banyak
        print(f"Unique string entries for column '{column}':\n{unique_entries}\n")

Terdapat 2 jenis kelamin, 4 program studi, 4 status mahasiswa, dan 5 nilai akhir yang berbeda. Hal ini menunjukkan perlu dilakukan encoding pada kolom-kolom tersebut agar dapat digunakan dalam model machine learning. Untuk nama, meskipun hanya terdapat 10 entri unik, ini hanya tanda identifikasi mahasiswa, sehingga tidak akan digunakan dalam pemodelan dan tidak perlu diencoding.

Berikut banyak baris per entri unik untuk jenis kelamin, prodi, status, dan nilai akhir.

In [ ]:
# Hitung banyak baris per entri unik
# Hanya untuk jenis kelamin, prodi, status, dan nilai akhir
columns = ['Jenis_Kelamin', 'Prodi', 'Status', 'Nilai_Akhir']

for column in columns:
    unique_counts = data[column].value_counts()
    print(f"Unique counts for column '{column}':\n{unique_counts}\nTotal: {unique_counts.sum()}\n")

Selain itu, dari lima kolom pertama yang ditampilkan, terdapat beberapa nilai yang hilang (NaN) pada kolom 'Nilai_Akhir' dan 'Umur'. Untuk mengetahui jumlah nilai yang hilang pada setiap kolom, kita dapat menggunakan fungsi `isnull()` dan `sum()`. Dan untuk melihat baris mana saja yang memiliki nilai yang hilang, kita dapat menggunakan fungsi `isnull()` dengan `any(axis=1)` untuk memfilter baris yang memiliki setidaknya satu nilai yang hilang.

In [ ]:
# Filter baris dengan nilai NA
na_rows = data[data.isnull().any(axis=1)]

# Hitung jumlah total nilai NA
total_cols_na = data.isnull().sum()
# Hitung baris dengan nilai NA
total_rows_na = na_rows.shape[0]

total_na = data.isnull().sum().sum()

# Cetak jumlah nilai NA per kolom
print("Count of NA values per column:")
print(total_cols_na[total_cols_na > 0])

# Cetak baris dengan nilai NA
print("Rows with NA values:")
display(na_rows.tail())

# Cetak jumlah total baris dengan nilai NA
print(f"Total count of rows with NA values: {total_rows_na}")

# Cetak jumlah total nilai NA
print(f"Total count of NA values: {total_na}")

Di sini terdapat 29 nilai kosong pada 'Nilai_Akhir' dan 9 nilai kosong pada 'Umur'. Hal ini menunjukkan bahwa kita perlu melakukan penanganan terhadap data yang hilang sebelum melanjutkan ke tahap pemodelan.

# Preprocessing Data {#sec-preprocessing}

Di sini, kita akan melakukan beberapa langkah preprocessing untuk menangani permasalahan yang ada dalam dataset, seperti data yang hilang, inkonsistensi format tanggal, dan encoding data kategorikal. Data yang diproses ini akan ditaruh dalam variabel `data_preprocessed` untuk digunakan dalam tahap pemodelan selanjutnya.

Pertama, kita akan membuat dataframe baru dan menyalin semua data dari data aslinya. Kolom 'ID' dipertahankan, karena kolom ini bertindak sebagai identifikasi dan tidak akan digunakan dalam pemodelan. Kolom-kolom lainnya akan diproses seperlunya. Dan untuk kolom 'Nama', ini akan dihapus karena kolom ini hanya sebagai pelengkap dan tidak akan digunakan dalam pemodelan.

In [ ]:
# Membuat dataframe baru untuk data yang sudah diproses
data_preprocessed = data.copy()

# Menghapus kolom 'Nama' karena tidak akan digunakan dalam pemodelan
data_preprocessed.drop(columns=['Nama'], inplace=True)

## Penanganan Data yang Hilang

Sebelumnya dalam @sec-eda, kita sudah mengetahui bahwa terdapat nilai yang hilang pada kolom 'Nilai_Akhir' dan 'Umur'. Untuk menangani data yang hilang, metode berikut akan digunakan:

- 'Nilai_Akhir': Nilai adalah data yang kritis. Melakukan pengisian nilai akan mengganggu integritas data, sehingga 29 baris dengan nilai yang hilang pada kolom ini akan dihapus.
- 'Umur': Data umur tidak terlalu kritis dan bisa diperkirakan berdasarkan data yang ada. Nilai yang hilang pada kolom ini akan diisi dengan nilai rata-rata umur dari data yang tersedia. Metode yang digunakan adalah `SimpleImputer` dari library `sklearn`, dengan strategi 'mean' untuk mengisi nilai yang hilang dengan rata-rata.

In [ ]:
# Untuk kolom 'Nilai_Akhir', nilai yang hilang akan dihapus
data_preprocessed = data_preprocessed.dropna(subset=['Nilai_Akhir'])

# Untuk kolom 'Umur', nilai yang hilang akan diisi dengan nilai rata-rata umur
imputer = SimpleImputer(strategy='mean')
data_preprocessed['Umur'] = imputer.fit_transform(data_preprocessed[['Umur']])

## Standarisasi Format Tanggal

Kolom 'Tanggal_Ujian' memiliki dua format tanggal yang berbeda, yaitu 'YYYY/MM/DD' dan 'DD-MM-YYYY'. Untuk memastikan konsistensi dalam pemrosesan data, kita akan mengubah semua format tanggal menjadi format yang sama sekaligus membuat kolom menjadi tipe data datetime.

Kita akan menggunakan fungsi `pd.to_datetime()` dari pandas untuk melakukan konversi ini. Dikarenakan terdapat dua format tanggal yang berbeda, kita akan menggunakan parameter `format='mixed'` untuk memungkinkan pandas mengenali kedua format tersebut. Selain itu, kita juga akan menggunakan parameter `yearfirst=True` dan `dayfirst=True` untuk memastikan bahwa pandas dapat mengidentifikasi bagian tahun dan hari dengan benar.

In [ ]:
# Mengubah format tanggal menjadi format yang sama dan mengonversi ke tipe datetime
data_preprocessed['Tanggal_Ujian'] = pd.to_datetime(data_preprocessed['Tanggal_Ujian'], errors='raise', format='mixed', yearfirst=True, dayfirst=True)

## Encoding Data Kategorikal

Encoding data kategorikal diperlukan agar data tersebut dapat digunakan dalam model machine learning, yang umumnya hanya dapat memproses data numerik. Berikut adalah beberapa asumsi terkait kolom-kolom kategorikal dalam dataset ini beserta metode encoding yang akan digunakan:

- **Jenis_Kelamin**: Terdapat dua kategori, yaitu 'Laki-laki' dan 'Perempuan'. Kita akan menggunakan encoding manual dengan memberikan nilai 0 untuk 'Laki-laki' dan 1 untuk 'Perempuan'.
- **Nilai_Akhir**: Terdapat lima kategori, yaitu 'A', 'B', 'C', 'D', dan 'E'. Data ini bersifat ordinal dan memiliki urutan mutu, sehingga kita akan menggunakan encoding manual berbobot dengan memberikan nilai 4 untuk 'A', 3 untuk 'B', 2 untuk 'C', 1 untuk 'D', dan 0 untuk 'E'.
- **Status**: Terdapat empat kategori, yaitu 'Aktif', 'Cuti', 'Lulus', dan 'DO'. Untuk mempermudah proses encoding, kita akan menggunakan *one-hot encoding*, yang akan menghasilkan empat kolom baru, masing-masing untuk setiap kategori status, dengan nilai 1 jika mahasiswa memiliki status tersebut dan 0 jika tidak.
- **Prodi**: Terdapat empat kategori, yaitu 'Informatika', 'Teknik Komputer', 'Data Science', dan 'Sistem Informasi'. Sama seperti 'Status', kita akan menggunakan *one-hot encoding* untuk kolom ini.

In [ ]:
# Kategori untuk jenis kelamin (manual)
gender_categories = {'Laki-laki': 0, 'Perempuan': 1}

# Kategori untuk nilai akhir (manual, berbobot)
grade_categories = {'A': 4, 'B': 3, 'C': 2, 'D': 1, 'E': 0}

Untuk encoding manual, operasi `map` sederhana akan digunakan. Sedangkan untuk melakukan *one-hot encoding*, kita akan menggunakan fungsi `get_dummies` dari pandas. Alternatif lain untuk melakukan *one-hot encoding* adalah dengan menggunakan `ColumnTransformer` dan `OneHotEncoder` dari library `sklearn`, namun prosesnya lebih kompleks, sehingga tidak akan digunakan di sini.

In [ ]:
# menggunakan pd.get_dummies() untuk melakukan one-hot encoding pada kolom 'Status' dan 'Prodi'
# fungsi ini juga membuat salinan data baru, sehingga kita bisa melakukan encoding manual pada kolom 'Jenis_Kelamin' dan 'Nilai_Akhir' tanpa mempengaruhi data asli
data_preprocessed_3 = pd.get_dummies(data_preprocessed, columns=['Status', 'Prodi'], prefix=['Status', 'Prodi'], drop_first=False, dtype=int)

# Melakukan encoding manual untuk kolom 'Jenis_Kelamin' dan 'Nilai_Akhir'
data_preprocessed_3['Jenis_Kelamin'] = data_preprocessed['Jenis_Kelamin'].map(gender_categories)
data_preprocessed_3['Nilai_Akhir'] = data_preprocessed['Nilai_Akhir'].map(grade_categories)

display(data_preprocessed_3[['ID', 'Jenis_Kelamin', 'Nilai_Akhir', 'Tanggal_Ujian', 'Umur', 'Status_Aktif', 'Prodi_Teknik Komputer']].head())

In [ ]:
#| echo: false
#| eval: false

# Selamat, anda menemukan Deleted Scenes! Berikut adalah contoh alternatif untuk melakukan one-hot encoding menggunakan ColumnTransformer dan OneHotEncoder dari sklearn. 

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Kolom yang akan diencoding dengan one-hot encoding
one_hot_columns = ['Status', 'Prodi']
# Membuat ColumnTransformer untuk melakukan one-hot encoding
column_transformer = ColumnTransformer(
    transformers=[
        ('onehot', OneHotEncoder(), one_hot_columns)  # drop='first' untuk menghindari dummy variable trap
    ],
    remainder='passthrough'  # Kolom lainnya tetap dipertahankan
)

# Melakukan encoding dengan ColumnTransformer
data_encoded = column_transformer.fit_transform(data_preprocessed)

# Membuat DataFrame baru dengan hasil encoding
data_preprocessed_2 = pd.DataFrame(data_encoded, columns=column_transformer.get_feature_names_out())

print(data_preprocessed_2.head())

In [ ]:
display(data_preprocessed_3.info())
display(data_preprocessed_3.describe())

# Pemodelan Machine Learning

Selanjutnya, kita akan melakukan pemodelan machine learning. Untuk pemodelan ini, kita akan menggunakan regresi linear untuk melakukan prediksi nilai akhir mahasiswa.

Sebelum melakukan pemodelan, data perlu dinormalisasi. Karena akan menggunakan regresi linear, maka metode normalisasi yang digunakan adalah *standarisasi* berdasarkan mean dan standar deviasi. 

Karena "Tanggal_Ujian" masih dalam bentuk datetime, perlu diubah menjadi bentuk numerik terlebih dahulu.

Untuk melakukan normalisasi, kita akan menggunakan fungsi `StandardScaler` dari library `sklearn`:

In [ ]:
from sklearn.preprocessing import StandardScaler

# Membuat objek StandardScaler
scaler = StandardScaler()

# Membuat salinan tabel data
data_premodel = data_preprocessed_3.copy().drop('ID', axis=1)

# Mengubah 'Tanggal_Ujian' menjadi bentuk numerik
data_premodel['Tanggal_Ujian'] = pd.to_numeric(data_premodel['Tanggal_Ujian'])

# Melakukan standarisasi pada data
data_premodel = scaler.fit_transform(data_premodel)

Langkah selanjutnya adalah membagi data menjadi data latih dan data uji. Data latih digunakan untuk melatih model, sedangkan data uji digunakan untuk menguji performa model yang telah dibuat. Dari soal, diminta untuk membagi data menjadi 80% untuk data latih dan 20% untuk data uji. Untuk membagi data, kita akan menggunakan fungsi `train_test_split` dari library `sklearn`. Untuk nilai y, kita akan menggunakan kolom 'Nilai_Akhir' sebagai target. Sedangkan untuk nilai X, kita akan menggunakan semua kolom selain 'Nilai_Akhir' sebagai fitur.

In [ ]:
from sklearn.model_selection import train_test_split

X = data_preprocessed_3.drop('Nilai_Akhir', axis=1)
y = data_preprocessed_3['Nilai_Akhir']

# Membagi data menjadi data latih dan data uji
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Menampilkan jumlah data latih dan data uji
print("Jumlah data latih:", len(X_train))
print("Jumlah data uji:", len(X_test))

Setelah melakukan normalisasi dan membagi data menjadi data latih dan data uji, kita akan melakukan pemodelan menggunakan regresi linear dengan fungsi `LinearRegression` dari library `sklearn`.

In [ ]:
from sklearn.linear_model import LinearRegression

# Membuat objek regresi linear
regressor = LinearRegression()

# Melatih model menggunakan data latih
regressor.fit(X_train, y_train)

# Evaluasi Model

# Visualisasi Hasil

# Kesimpulan

# Lampiran {#sec-appendix}

## Diagnostik Python {.appendix}

In [ ]:
import IPython as ipy
print(ipy.sys_info())